#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

#`Setup And Update ComfyUI`



In [ ]:
from pathlib import Path
import os

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
COMFYUI_LAUNCH_ARGS = "--dont-print-server --lowvram --disable-auto-launch"  #@param {type:"string"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

os.environ["PIP_CACHE_DIR"] = "/content/pip-cache"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

if DRIVE_PATH:

    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

if not Path(WORKSPACE).exists():
  !echo -= Initial setup ComfyUI =-
  !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
%cd {WORKSPACE}

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install -q --upgrade pip
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}


# `Models Download`

## Download MODELS

In [ ]:
%cd {WORKSPACE}
!pip install -q -U "huggingface_hub[cli]" hf_transfer

import os
import subprocess
from pathlib import Path
from google.colab import userdata

os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

HF_TOKEN = (
    userdata.get("HF_TOKEN")
    or userdata.get("HUGGINGFACE_TOKEN")
    or os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
)
if not HF_TOKEN:
    raise ValueError("Add a Colab secret named HF_TOKEN with access to the requested Hugging Face repos.")
print("Loaded Hugging Face token: yes")

MODEL_DIRS = {
    "diffusion": Path(WORKSPACE) / "models" / "diffusion_models",
    "text_encoder": Path(WORKSPACE) / "models" / "text_encoders",
    "vae": Path(WORKSPACE) / "models" / "vae",
}
for model_dir in MODEL_DIRS.values():
    model_dir.mkdir(parents=True, exist_ok=True)
(Path(WORKSPACE) / "custom_nodes").mkdir(parents=True, exist_ok=True)


def hf_download(repo_id, filename, local_dir, target_name=None):
    local_dir = Path(local_dir)
    target = local_dir / (target_name or Path(filename).name)
    if target.exists() and target.stat().st_size > 0:
        print(f"Already present: {target}")
        return target

    cmd = [
        "hf",
        "download",
        repo_id,
        filename,
        "--local-dir",
        str(local_dir),
        "--token",
        HF_TOKEN,
    ]
    env = os.environ.copy()
    result = subprocess.run(cmd, env=env, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"hf download failed for {repo_id}/{filename}")

    downloaded = local_dir / filename
    if target_name and downloaded.exists() and downloaded != target:
        target.parent.mkdir(parents=True, exist_ok=True)
        downloaded.replace(target)
        parent = downloaded.parent
        while parent != local_dir:
            try:
                parent.rmdir()
            except OSError:
                break
            parent = parent.parent
    print(f"Ready: {target}")
    return target

hf_download(
    "Dervlex/Private",
    "VeniceV2_bf16.safetensors",
    MODEL_DIRS["diffusion"],
)
hf_download(
    "ponpoke/flux2-klein-9b-uncensored-text-encoder",
    "flux2-klein-9b-uncensored-q8_0.gguf",
    MODEL_DIRS["text_encoder"],
)
hf_download(
    "black-forest-labs/FLUX.2-klein-9B",
    "vae/diffusion_pytorch_model.safetensors",
    MODEL_DIRS["vae"],
    target_name="FLUX.2-klein-9B-vae.safetensors",
)

%cd {WORKSPACE}


## INSTALL CUSTOM NODES


In [ ]:
from collections import OrderedDict
from pathlib import Path
import subprocess
import sys

COMFYUI_PATH = Path("/content/ComfyUI")
CUSTOM_NODES_PATH = COMFYUI_PATH / "custom_nodes"
CUSTOM_NODES_PATH.mkdir(parents=True, exist_ok=True)

custom_node_repos = OrderedDict((
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF.git"),
    ("ComfyUI-Easy-Use", "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("ComfyUI-Crystools", "https://github.com/crystian/ComfyUI-Crystools.git"),
    ("ComfyUI-Lora-Manager", "https://github.com/jthickma/ComfyUI-Lora-Manager.git"),
))

for name, url in custom_node_repos.items():
    node_path = CUSTOM_NODES_PATH / name
    if node_path.exists():
        print(f"Updating {name}")
        subprocess.run(["git", "-C", str(node_path), "pull", "--ff-only"], check=True)
    else:
        print(f"Cloning {name}")
        subprocess.run(["git", "clone", url, str(node_path)], check=True)

    requirements = node_path / "requirements.txt"
    if requirements.exists():
        print(f"Installing requirements for {name}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
    else:
        print(f"No requirements.txt for {name}")

%cd /content/ComfyUI


## COLAB OPTIMIZATIONS


In [ ]:
# Colab optimization notes
# - Keep DRIVE_PATH empty for fastest ephemeral installs; set it only when you need model persistence.
# - HF transfers are accelerated with hf_transfer and cached under /content/hf-cache to avoid slow Drive metadata churn.
# - COMFYUI_LAUNCH_ARGS defaults to --lowvram for Colab GPUs; remove it on high-VRAM runtimes if you want more speed.
# - The custom-node cell installs ComfyUI-GGUF for workflows that load GGUF text encoders.


## LIST MODELS

In [ ]:
%cd {WORKSPACE}
!find ./models/diffusion_models ./models/text_encoders ./models/vae -maxdepth 2 -type f -print -exec ls -lh {} \;


# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!python main.py {COMFYUI_LAUNCH_ARGS}


# `START ComfyUI & Expose Server`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## CF Tunnel

In [ ]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py {COMFYUI_LAUNCH_ARGS}


## localtunnel

In [ ]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd {WORKSPACE}
!python main.py {COMFYUI_LAUNCH_ARGS}
